# 5.5 Applications of Inner Product Spaces

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 5 — Inner Product Spaces**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.inner import dot, norm, cross_product
from linalg_core.determinant import det_cofactor, det_elimination
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import math

print("Setup complete.")

---

## 2. Motivation

A mechanic tightens a bolt using a wrench. She applies a **force** $\mathbf{F}$ at the end
of a handle whose position relative to the bolt is described by a **lever arm** vector
$\mathbf{r}$. The **torque** — the rotational force that actually turns the bolt — is

$$\boldsymbol{\tau} = \mathbf{r} \times \mathbf{F}$$

This simple formula encodes three pieces of information at once:
- **Direction:** the axis of rotation (perpendicular to both $\mathbf{r}$ and $\mathbf{F}$).
- **Magnitude:** how much rotational force is applied ($\|\boldsymbol{\tau}\| = \|\mathbf{r}\| \, \|\mathbf{F}\| \sin\theta$).
- **Sense:** which way the bolt turns (right-hand rule).

But what exactly *is* the **cross product** $\mathbf{u} \times \mathbf{v}$, and how
do we compute it? That is the subject of this notebook.

The cross product connects several themes from this course:
- Inner products (it is perpendicular to both inputs).
- Norms (its magnitude equals the area of a parallelogram).
- Determinants (it can be expressed as a formal $3 \times 3$ determinant).

We will build it from scratch, verify every property, and apply it to
geometry and physics.

---

## 3. Build — Core Concepts

### 3.1 The Cross Product Formula

> **Definition (Larson, Sec. 5.5).** The **cross product** of two vectors
> $\mathbf{u} = (u_1, u_2, u_3)$ and $\mathbf{v} = (v_1, v_2, v_3)$ in $\mathbb{R}^3$ is
>
> $$\mathbf{u} \times \mathbf{v} = \begin{bmatrix} u_2 v_3 - u_3 v_2 \\ u_3 v_1 - u_1 v_3 \\ u_1 v_2 - u_2 v_1 \end{bmatrix}$$

Each component is a $2 \times 2$ determinant built from the entries of $\mathbf{u}$
and $\mathbf{v}$ with one coordinate "crossed out":

$$
(\mathbf{u} \times \mathbf{v})_1 = \det\begin{bmatrix} u_2 & u_3 \\ v_2 & v_3 \end{bmatrix}, \qquad
(\mathbf{u} \times \mathbf{v})_2 = -\det\begin{bmatrix} u_1 & u_3 \\ v_1 & v_3 \end{bmatrix}, \qquad
(\mathbf{u} \times \mathbf{v})_3 = \det\begin{bmatrix} u_1 & u_2 \\ v_1 & v_2 \end{bmatrix}
$$

**Important:** The cross product is defined **only in $\mathbb{R}^3$**.
It takes two 3D vectors and returns another 3D vector.

Our library provides `cross_product(u, v)` which implements exactly this formula.

In [ ]:
u = [1, 2, 3]
v = [4, 5, 6]

print(f"u = {u}")
print(f"v = {v}")

print("\n--- Cross product by hand ---")
c1 = u[1]*v[2] - u[2]*v[1]
c2 = u[2]*v[0] - u[0]*v[2]
c3 = u[0]*v[1] - u[1]*v[0]
print(f"Component 1: u2*v3 - u3*v2 = ({u[1]})*({v[2]}) - ({u[2]})*({v[1]}) = {c1}")
print(f"Component 2: u3*v1 - u1*v3 = ({u[2]})*({v[0]}) - ({u[0]})*({v[2]}) = {c2}")
print(f"Component 3: u1*v2 - u2*v1 = ({u[0]})*({v[1]}) - ({u[1]})*({v[0]}) = {c3}")
print(f"u x v (by hand) = [{c1}, {c2}, {c3}]")

print("\n--- Using cross_product ---")
result = cross_product(u, v)
print(f"cross_product(u, v) = {result}")

print(f"\nMatch: {result == [c1, c2, c3]}")

### 3.2 Properties of the Cross Product

The cross product has several important algebraic properties that distinguish it
from other operations we have seen:

| Property | Statement | Contrast |
|---|---|---|
| **Anticommutative** | $\mathbf{u} \times \mathbf{v} = -(\mathbf{v} \times \mathbf{u})$ | Dot product is commutative |
| **Not associative** | $(\mathbf{u} \times \mathbf{v}) \times \mathbf{w} \neq \mathbf{u} \times (\mathbf{v} \times \mathbf{w})$ in general | Scalar multiplication is associative |
| **Distributive** | $\mathbf{u} \times (\mathbf{v} + \mathbf{w}) = \mathbf{u} \times \mathbf{v} + \mathbf{u} \times \mathbf{w}$ | Same as dot product |
| **Self-cross is zero** | $\mathbf{u} \times \mathbf{u} = \mathbf{0}$ | $\mathbf{u} \cdot \mathbf{u} = \|\mathbf{u}\|^2 \neq 0$ |
| **Scalar factor** | $(c\mathbf{u}) \times \mathbf{v} = c(\mathbf{u} \times \mathbf{v})$ | Same as dot product |

The anticommutativity and non-associativity are the most surprising. Swapping
the order **reverses the direction**, and parentheses **matter**.

Let us verify each property numerically.

In [ ]:
u = [1, 2, 3]
v = [4, 5, 6]
w = [-1, 3, 2]

def vec_eq(a, b, tol=EPSILON):
    """Check if two vectors are equal within tolerance."""
    return all(abs(x - y) < tol for x, y in zip(a, b))

def vec_neg(a):
    """Negate a vector."""
    return [-x for x in a]

def vec_add(a, b):
    """Add two vectors."""
    return [x + y for x, y in zip(a, b)]

def vec_scale(c, a):
    """Scale a vector."""
    return [c * x for x in a]

print("=" * 55)
print("Property 1: Anticommutative  u x v = -(v x u)")
print("=" * 55)
uv = cross_product(u, v)
vu = cross_product(v, u)
neg_vu = vec_neg(vu)
print(f"u x v     = {uv}")
print(f"v x u     = {vu}")
print(f"-(v x u)  = {neg_vu}")
print(f"u x v == -(v x u): {vec_eq(uv, neg_vu)}")

print(f"\n{'=' * 55}")
print("Property 2: NOT associative  (u x v) x w != u x (v x w)")
print("=" * 55)
lhs = cross_product(cross_product(u, v), w)
rhs = cross_product(u, cross_product(v, w))
print(f"(u x v) x w = {lhs}")
print(f"u x (v x w) = {rhs}")
print(f"Are they equal? {vec_eq(lhs, rhs)}")
print("The cross product is NOT associative!")

print(f"\n{'=' * 55}")
print("Property 3: Distributive  u x (v + w) = u x v + u x w")
print("=" * 55)
lhs = cross_product(u, vec_add(v, w))
rhs = vec_add(cross_product(u, v), cross_product(u, w))
print(f"u x (v + w)       = {lhs}")
print(f"u x v + u x w     = {rhs}")
print(f"Equal: {vec_eq(lhs, rhs)}")

print(f"\n{'=' * 55}")
print("Property 4: Self-cross is zero  u x u = 0")
print("=" * 55)
uu = cross_product(u, u)
print(f"u x u = {uu}")
print(f"Is zero vector: {vec_eq(uu, [0, 0, 0])}")

print(f"\n{'=' * 55}")
print("Property 5: Scalar factor  (cu) x v = c(u x v)")
print("=" * 55)
c = 3
lhs = cross_product(vec_scale(c, u), v)
rhs = vec_scale(c, cross_product(u, v))
print(f"({c}u) x v  = {lhs}")
print(f"{c}(u x v)  = {rhs}")
print(f"Equal: {vec_eq(lhs, rhs)}")

### 3.3 Geometric Meaning — Perpendicularity

> **Theorem (Larson, Sec. 5.5).** The cross product $\mathbf{u} \times \mathbf{v}$
> is **orthogonal** to both $\mathbf{u}$ and $\mathbf{v}$.

**Proof sketch.** Compute $\mathbf{u} \cdot (\mathbf{u} \times \mathbf{v})$ directly
from the formulas:

$$
\mathbf{u} \cdot (\mathbf{u} \times \mathbf{v})
= u_1(u_2 v_3 - u_3 v_2) + u_2(u_3 v_1 - u_1 v_3) + u_3(u_1 v_2 - u_2 v_1)
$$

Expanding and grouping, every term cancels:

$$= u_1 u_2 v_3 - u_1 u_3 v_2 + u_2 u_3 v_1 - u_1 u_2 v_3 + u_1 u_3 v_2 - u_2 u_3 v_1 = 0$$

The same argument works for $\mathbf{v} \cdot (\mathbf{u} \times \mathbf{v}) = 0$. $\square$

This is the key geometric fact: the cross product gives the **normal direction**
to the plane containing $\mathbf{u}$ and $\mathbf{v}$.

The **magnitude** of the cross product also has geometric meaning:

$$\|\mathbf{u} \times \mathbf{v}\| = \|\mathbf{u}\| \, \|\mathbf{v}\| \, \sin\theta$$

where $\theta$ is the angle between $\mathbf{u}$ and $\mathbf{v}$. This equals
the **area of the parallelogram** spanned by $\mathbf{u}$ and $\mathbf{v}$.

In [ ]:
u = [1, 2, 3]
v = [4, 5, 6]

cross = cross_product(u, v)
print(f"u = {u}")
print(f"v = {v}")
print(f"u x v = {cross}")

print("\n--- Orthogonality check ---")
dot_u_cross = dot(u, cross)
dot_v_cross = dot(v, cross)
print(f"u . (u x v) = {dot_u_cross:.10f}")
print(f"v . (u x v) = {dot_v_cross:.10f}")
print(f"u perpendicular to (u x v): {abs(dot_u_cross) < EPSILON}")
print(f"v perpendicular to (u x v): {abs(dot_v_cross) < EPSILON}")

print("\n--- Magnitude = ||u|| ||v|| sin(theta) ---")
norm_cross = norm(cross)
norm_u = norm(u)
norm_v = norm(v)
cos_theta = dot(u, v) / (norm_u * norm_v)
sin_theta = math.sqrt(1 - cos_theta**2)
expected_magnitude = norm_u * norm_v * sin_theta
print(f"||u x v|| = {norm_cross:.6f}")
print(f"||u|| * ||v|| * sin(theta) = {norm_u:.4f} * {norm_v:.4f} * {sin_theta:.6f} = {expected_magnitude:.6f}")
print(f"Match: {abs(norm_cross - expected_magnitude) < EPSILON}")

### 3.4 Parallelogram Area

> **Theorem (Larson, Sec. 5.5).** The area of the parallelogram with adjacent
> sides $\mathbf{u}$ and $\mathbf{v}$ is
>
> $$\text{Area} = \|\mathbf{u} \times \mathbf{v}\|$$

Recall that a parallelogram with sides of length $a$ and $b$ and included angle
$\theta$ has area $ab \sin\theta$. Since $\|\mathbf{u} \times \mathbf{v}\| = \|\mathbf{u}\|\,\|\mathbf{v}\|\sin\theta$,
the cross product magnitude gives us the area directly.

**Example.** Find the area of the parallelogram with adjacent sides
$\mathbf{u} = (3, 2, -1)$ and $\mathbf{v} = (1, -1, 4)$.

In [ ]:
u = [3, 2, -1]
v = [1, -1, 4]

cross = cross_product(u, v)
area = norm(cross)

print(f"u = {u}")
print(f"v = {v}")
print(f"\nu x v = {cross}")

print("\n--- Step by step ---")
print(f"Component 1: ({u[1]})*({v[2]}) - ({u[2]})*({v[1]}) = {u[1]*v[2]} - ({u[2]*v[1]}) = {cross[0]}")
print(f"Component 2: ({u[2]})*({v[0]}) - ({u[0]})*({v[2]}) = {u[2]*v[0]} - {u[0]*v[2]} = {cross[1]}")
print(f"Component 3: ({u[0]})*({v[1]}) - ({u[1]})*({v[0]}) = {u[0]*v[1]} - {u[1]*v[0]} = {cross[2]}")

print(f"\n||u x v|| = sqrt({cross[0]}^2 + {cross[1]}^2 + {cross[2]}^2)")
print(f"         = sqrt({cross[0]**2} + {cross[1]**2} + {cross[2]**2})")
print(f"         = sqrt({cross[0]**2 + cross[1]**2 + cross[2]**2})")
print(f"         = {area:.6f}")

print(f"\nParallelogram area = {area:.6f}")

print("\n--- Verification with ||u|| ||v|| sin(theta) ---")
norm_u = norm(u)
norm_v = norm(v)
cos_theta = dot(u, v) / (norm_u * norm_v)
sin_theta = math.sqrt(1 - cos_theta**2)
area_alt = norm_u * norm_v * sin_theta
print(f"||u|| = {norm_u:.6f}")
print(f"||v|| = {norm_v:.6f}")
print(f"sin(theta) = {sin_theta:.6f}")
print(f"||u|| * ||v|| * sin(theta) = {area_alt:.6f}")
print(f"Match: {abs(area - area_alt) < EPSILON}")

### 3.5 Parallelepiped Volume — The Triple Scalar Product

A **parallelepiped** is the 3D analogue of a parallelogram — a slanted box whose
edges are defined by three vectors $\mathbf{u}$, $\mathbf{v}$, $\mathbf{w}$.

> **Theorem (Larson, Sec. 5.5).** The volume of the parallelepiped with adjacent
> edges $\mathbf{u}$, $\mathbf{v}$, $\mathbf{w}$ is
>
> $$V = |\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w})|$$

The quantity $\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w})$ is called the
**triple scalar product** (or **scalar triple product**).

**Geometric interpretation:**
- $\mathbf{v} \times \mathbf{w}$ gives a vector perpendicular to the base parallelogram,
  whose magnitude equals the base area.
- Dotting with $\mathbf{u}$ projects $\mathbf{u}$ onto the normal direction, giving the
  **signed height**.
- Volume = base area $\times$ height, so $V = |\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w})|$.

**Example.** Find the volume of the parallelepiped with edges
$\mathbf{u} = (1, 0, 2)$, $\mathbf{v} = (0, 1, -1)$, $\mathbf{w} = (3, 1, 0)$.

In [ ]:
u = [1, 0, 2]
v = [0, 1, -1]
w = [3, 1, 0]

print(f"u = {u}")
print(f"v = {v}")
print(f"w = {w}")

print("\n--- Step 1: Compute v x w ---")
vw = cross_product(v, w)
print(f"v x w = {vw}")

print("\n--- Step 2: Compute u . (v x w) ---")
triple = dot(u, vw)
print(f"u . (v x w) = {u[0]}*{vw[0]} + {u[1]}*{vw[1]} + {u[2]}*{vw[2]}")
print(f"            = {u[0]*vw[0]} + {u[1]*vw[1]} + {u[2]*vw[2]}")
print(f"            = {triple}")

print("\n--- Step 3: Volume = |triple scalar product| ---")
volume = abs(triple)
print(f"Volume = |{triple}| = {volume}")

print("\n--- Interpretation ---")
base_area = norm(vw)
print(f"Base area (||v x w||) = {base_area:.6f}")
if base_area > EPSILON:
    height = volume / base_area
    print(f"Height (V / base area) = {height:.6f}")
    print(f"Volume = base_area * height = {base_area:.6f} * {height:.6f} = {base_area * height:.6f}")

### 3.6 Application: Torque

Returning to the motivating problem: a wrench of length 0.3 m points along
$\mathbf{r} = (0.3, 0, 0)$ (horizontal). A force of 50 N is applied at the end,
directed at $30^\circ$ below the horizontal plane:

$$\mathbf{F} = (0, \, -50\sin 30^\circ, \, -50\cos 30^\circ) = (0, \, -25, \, -25\sqrt{3})$$

The torque is:

$$\boldsymbol{\tau} = \mathbf{r} \times \mathbf{F}$$

The magnitude $\|\boldsymbol{\tau}\|$ tells us how much rotational force is applied
(in Newton-meters), and the direction tells us the axis of rotation.

In [ ]:
r = [0.3, 0, 0]
F = [0, -25, -25 * math.sqrt(3)]

print(f"Lever arm:  r = [{r[0]:.1f}, {r[1]:.1f}, {r[2]:.1f}] m")
print(f"Force:      F = [{F[0]:.1f}, {F[1]:.1f}, {F[2]:.4f}] N")

tau = cross_product(r, F)
print(f"\nTorque: tau = r x F = [{tau[0]:.6f}, {tau[1]:.6f}, {tau[2]:.6f}]")

tau_magnitude = norm(tau)
print(f"\n|tau| = {tau_magnitude:.6f} N*m")

print("\n--- Physical interpretation ---")
print(f"Torque magnitude: {tau_magnitude:.2f} N*m")
print(f"Torque direction: [{tau[0]:.4f}, {tau[1]:.4f}, {tau[2]:.4f}]")

if abs(tau[0]) > EPSILON:
    print(f"  -> Nonzero x-component means rotation around the x-axis")
if abs(tau[1]) > EPSILON:
    print(f"  -> Nonzero y-component: tau_y = {tau[1]:.4f} N*m")
if abs(tau[2]) > EPSILON:
    print(f"  -> Nonzero z-component: tau_z = {tau[2]:.4f} N*m")

print("\n--- Verification: |tau| = |r| * |F| * sin(theta) ---")
norm_r = norm(r)
norm_F = norm(F)
cos_theta = dot(r, F) / (norm_r * norm_F)
sin_theta = math.sqrt(1 - cos_theta**2)
expected_mag = norm_r * norm_F * sin_theta
print(f"|r| = {norm_r:.4f} m")
print(f"|F| = {norm_F:.4f} N")
print(f"sin(theta) = {sin_theta:.6f}")
print(f"|r| * |F| * sin(theta) = {expected_mag:.6f} N*m")
print(f"Match: {abs(tau_magnitude - expected_mag) < EPSILON}")

### 3.7 Connection to Determinants

The cross product can be written as a **formal determinant** using the standard
basis vectors $\mathbf{i}, \mathbf{j}, \mathbf{k}$:

$$
\mathbf{u} \times \mathbf{v} = \det\begin{bmatrix}
\mathbf{i} & \mathbf{j} & \mathbf{k} \\
u_1 & u_2 & u_3 \\
v_1 & v_2 & v_3
\end{bmatrix}
$$

Expanding along the first row by cofactors:

$$
= \mathbf{i}\det\begin{bmatrix} u_2 & u_3 \\ v_2 & v_3 \end{bmatrix}
- \mathbf{j}\det\begin{bmatrix} u_1 & u_3 \\ v_1 & v_3 \end{bmatrix}
+ \mathbf{k}\det\begin{bmatrix} u_1 & u_2 \\ v_1 & v_2 \end{bmatrix}
$$

$$
= (u_2 v_3 - u_3 v_2)\,\mathbf{i} - (u_1 v_3 - u_3 v_1)\,\mathbf{j} + (u_1 v_2 - u_2 v_1)\,\mathbf{k}
$$

which is exactly our cross product formula.

This is called a **formal** determinant because the first row contains vectors,
not scalars — but the cofactor expansion procedure still produces the correct formula.

The **triple scalar product** also connects to determinants:

$$\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w}) = \det\begin{bmatrix}
u_1 & u_2 & u_3 \\
v_1 & v_2 & v_3 \\
w_1 & w_2 & w_3
\end{bmatrix}$$

So the volume of a parallelepiped equals the **absolute value of a determinant**!

In [ ]:
u = [1, 2, 3]
v = [4, 5, 6]

print("=" * 60)
print("Cross product as formal determinant")
print("=" * 60)
print(f"\nu = {u}")
print(f"v = {v}")

print("\nFormal determinant:")
print("| i   j   k  |")
print(f"| {u[0]}   {u[1]}   {u[2]}  |")
print(f"| {v[0]}   {v[1]}   {v[2]}  |")

cofactor_i = u[1]*v[2] - u[2]*v[1]
cofactor_j = -(u[0]*v[2] - u[2]*v[0])
cofactor_k = u[0]*v[1] - u[1]*v[0]

print(f"\nCofactor of i: det[[{u[1]},{u[2]}],[{v[1]},{v[2]}]] = {cofactor_i}")
print(f"Cofactor of j: -det[[{u[0]},{u[2]}],[{v[0]},{v[2]}]] = {cofactor_j}")
print(f"Cofactor of k: det[[{u[0]},{u[1]}],[{v[0]},{v[1]}]] = {cofactor_k}")

det_result = [cofactor_i, cofactor_j, cofactor_k]
cross_result = cross_product(u, v)
print(f"\nFrom determinant: {det_result}")
print(f"From cross_product: {cross_result}")
print(f"Match: {det_result == cross_result}")

print(f"\n{'=' * 60}")
print("Triple scalar product = determinant of 3x3 matrix")
print("=" * 60)

u = [1, 0, 2]
v = [0, 1, -1]
w = [3, 1, 0]

print(f"\nu = {u}")
print(f"v = {v}")
print(f"w = {w}")

triple = dot(u, cross_product(v, w))
print(f"\nu . (v x w) = {triple}")

M = Matrix.from_lists([u, v, w])
det_val = det_cofactor(M)
print(f"\nMatrix [u; v; w] =")
print(M)
print(f"\ndet([u; v; w]) = {det_val:.6f}")
print(f"\nTriple scalar product == det: {abs(triple - det_val) < EPSILON}")
print(f"\nParallelepiped volume = |det| = {abs(det_val):.6f}")

---

## 4. Verify — Cross-Check with NumPy

Our from-scratch `cross_product` should agree with `np.cross` for every vector pair.
We also verify the geometric properties: orthogonality, magnitude = parallelogram area,
and triple scalar product = determinant.

In [ ]:
import random
random.seed(42)

def check_cross(u, v, label):
    """Compare cross_product vs np.cross and check orthogonality."""
    ours = cross_product(u, v)
    theirs = list(np.cross(u, v))
    match_np = all(abs(a - b) < 1e-9 for a, b in zip(ours, theirs))

    ortho_u = abs(dot(u, ours)) < 1e-9
    ortho_v = abs(dot(v, ours)) < 1e-9

    all_ok = match_np and ortho_u and ortho_v
    status = "PASS" if all_ok else "FAIL"
    print(f"[{status}] {label}: np_match={match_np}, ortho_u={ortho_u}, ortho_v={ortho_v}")
    return all_ok

In [ ]:
print("=" * 60)
print("VERIFICATION: cross_product vs np.cross + orthogonality")
print("=" * 60)

test_pairs = [
    ([1, 0, 0], [0, 1, 0], "i x j = k"),
    ([0, 1, 0], [0, 0, 1], "j x k = i"),
    ([0, 0, 1], [1, 0, 0], "k x i = j"),
    ([1, 2, 3], [4, 5, 6], "general #1"),
    ([3, 2, -1], [1, -1, 4], "general #2"),
    ([1, 0, 2], [0, 1, -1], "general #3"),
    ([2, 2, 2], [1, 1, 1], "parallel vectors"),
]

all_pass = True
for u, v, label in test_pairs:
    if not check_cross(u, v, label):
        all_pass = False

print("\n--- Random tests ---")
for i in range(10):
    u = [random.uniform(-10, 10) for _ in range(3)]
    v = [random.uniform(-10, 10) for _ in range(3)]
    if not check_cross(u, v, f"random #{i+1}"):
        all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION: ||u x v|| = parallelogram area")
print("=" * 60)

all_pass = True
for i in range(10):
    u = [random.uniform(-10, 10) for _ in range(3)]
    v = [random.uniform(-10, 10) for _ in range(3)]
    cross = cross_product(u, v)
    norm_cross = norm(cross)

    norm_u = norm(u)
    norm_v = norm(v)
    if norm_u > EPSILON and norm_v > EPSILON:
        cos_t = max(-1.0, min(1.0, dot(u, v) / (norm_u * norm_v)))
        sin_t = math.sqrt(1 - cos_t**2)
        expected = norm_u * norm_v * sin_t
        match = abs(norm_cross - expected) < 1e-6
        status = "PASS" if match else "FAIL"
        print(f"[{status}] Trial {i+1}: ||cross||={norm_cross:.6f}, area={expected:.6f}")
        if not match:
            all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION: triple scalar product = det of 3x3 matrix")
print("=" * 60)

all_pass = True
for i in range(10):
    u = [random.uniform(-10, 10) for _ in range(3)]
    v = [random.uniform(-10, 10) for _ in range(3)]
    w = [random.uniform(-10, 10) for _ in range(3)]

    triple = dot(u, cross_product(v, w))
    M = Matrix.from_lists([u, v, w])
    det_val = det_cofactor(M)
    det_val2 = det_elimination(M)

    match_cof = abs(triple - det_val) < 1e-6
    match_elim = abs(triple - det_val2) < 1e-6
    status = "PASS" if (match_cof and match_elim) else "FAIL"
    print(f"[{status}] Trial {i+1}: triple={triple:.4f}, det_cofactor={det_val:.4f}, det_elim={det_val2:.4f}")
    if not (match_cof and match_elim):
        all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

---

## 5. Visualize

### 5.1 Two Vectors and Their Cross Product

The cross product is perpendicular to both input vectors. We draw all three
from the origin and shade the parallelogram they span.

In [ ]:
u = [2, 1, 0]
v = [0, 2, 1]
cross = cross_product(u, v)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

origin = [0, 0, 0]

ax.quiver(*origin, *u, color='blue', linewidth=2.5, arrow_length_ratio=0.1, label=f'u = {u}')
ax.quiver(*origin, *v, color='red', linewidth=2.5, arrow_length_ratio=0.1, label=f'v = {v}')
ax.quiver(*origin, *cross, color='green', linewidth=2.5, arrow_length_ratio=0.1,
          label=f'u x v = {cross}')

parallelogram = [
    [0, 0, 0],
    u,
    [u[0]+v[0], u[1]+v[1], u[2]+v[2]],
    v
]
poly = Poly3DCollection([parallelogram], alpha=0.2, facecolor='purple', edgecolor='purple', linewidth=1)
ax.add_collection3d(poly)

all_coords = [origin, u, v, cross, [u[0]+v[0], u[1]+v[1], u[2]+v[2]]]
xs = [p[0] for p in all_coords]
ys = [p[1] for p in all_coords]
zs = [p[2] for p in all_coords]
margin = 0.5
ax.set_xlim([min(xs) - margin, max(xs) + margin])
ax.set_ylim([min(ys) - margin, max(ys) + margin])
ax.set_zlim([min(zs) - margin, max(zs) + margin])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Cross Product: u x v is perpendicular to both u and v')
ax.legend(loc='upper left')
ax.view_init(elev=20, azim=135)
plt.tight_layout()
plt.show()

print(f"\nParallelogram area = ||u x v|| = {norm(cross):.4f}")
print(f"u . (u x v) = {dot(u, cross):.10f}  (orthogonal)")
print(f"v . (u x v) = {dot(v, cross):.10f}  (orthogonal)")

### 5.2 Parallelogram Area Visualization

We draw the parallelogram spanned by $\mathbf{u}$ and $\mathbf{v}$ viewed from
a direction that emphasizes the area, and annotate it.

In [ ]:
u = [3, 1, 0]
v = [1, 3, 0]
cross = cross_product(u, v)
area = norm(cross)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.quiver(0, 0, 0, *u, color='blue', linewidth=2.5, arrow_length_ratio=0.08, label=f'u = {u}')
ax.quiver(0, 0, 0, *v, color='red', linewidth=2.5, arrow_length_ratio=0.08, label=f'v = {v}')

scale = 0.3
cross_scaled = [scale * c for c in cross]
mid = [(u[i] + v[i]) / 2 for i in range(3)]
ax.quiver(*mid, *cross_scaled, color='green', linewidth=2, arrow_length_ratio=0.15,
          label=f'normal (scaled)')

parallelogram = [
    [0, 0, 0],
    u,
    [u[0]+v[0], u[1]+v[1], u[2]+v[2]],
    v
]
poly = Poly3DCollection([parallelogram], alpha=0.35, facecolor='gold', edgecolor='black', linewidth=2)
ax.add_collection3d(poly)

ax.text(mid[0], mid[1], mid[2] - 0.3, f'Area = {area:.2f}',
        fontsize=14, fontweight='bold', ha='center', color='darkred')

ax.set_xlim([-0.5, 5])
ax.set_ylim([-0.5, 5])
ax.set_zlim([-1, 3])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'Parallelogram Area = ||u x v|| = {area:.2f}')
ax.legend(loc='upper left')
ax.view_init(elev=55, azim=135)
plt.tight_layout()
plt.show()

### 5.3 Parallelepiped Volume Visualization

A parallelepiped is the 3D "slanted box" formed by three edge vectors.
Its volume is $|\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w})|$.

In [ ]:
u = [2, 0, 0]
v = [0.5, 2, 0]
w = [0.3, 0.3, 1.5]

volume = abs(dot(u, cross_product(v, w)))

vertices = [
    [0, 0, 0],
    u,
    v,
    w,
    [u[i]+v[i] for i in range(3)],
    [u[i]+w[i] for i in range(3)],
    [v[i]+w[i] for i in range(3)],
    [u[i]+v[i]+w[i] for i in range(3)],
]

O = vertices[0]
U = vertices[1]
V = vertices[2]
W = vertices[3]
UV = vertices[4]
UW = vertices[5]
VW = vertices[6]
UVW = vertices[7]

faces = [
    [O, U, UV, V],
    [O, U, UW, W],
    [O, V, VW, W],
    [U, UV, UVW, UW],
    [V, UV, UVW, VW],
    [W, UW, UVW, VW],
]

face_colors = ['cyan', 'yellow', 'magenta', 'cyan', 'yellow', 'magenta']

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for face, color in zip(faces, face_colors):
    poly = Poly3DCollection([face], alpha=0.15, facecolor=color, edgecolor='black', linewidth=1)
    ax.add_collection3d(poly)

ax.quiver(0, 0, 0, *u, color='blue', linewidth=2.5, arrow_length_ratio=0.06, label=f'u = {u}')
ax.quiver(0, 0, 0, *v, color='red', linewidth=2.5, arrow_length_ratio=0.06, label=f'v = {v}')
ax.quiver(0, 0, 0, *w, color='green', linewidth=2.5, arrow_length_ratio=0.06, label=f'w = {w}')

center = [(u[i]+v[i]+w[i]) / 2 for i in range(3)]
ax.text(center[0], center[1], center[2] + 0.3, f'V = {volume:.2f}',
        fontsize=14, fontweight='bold', ha='center', color='darkblue')

all_pts = vertices
xs = [p[0] for p in all_pts]
ys = [p[1] for p in all_pts]
zs = [p[2] for p in all_pts]
margin = 0.5
ax.set_xlim([min(xs) - margin, max(xs) + margin])
ax.set_ylim([min(ys) - margin, max(ys) + margin])
ax.set_zlim([min(zs) - margin, max(zs) + margin])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'Parallelepiped: Volume = |u . (v x w)| = {volume:.2f}')
ax.legend(loc='upper left')
ax.view_init(elev=25, azim=135)
plt.tight_layout()
plt.show()

print(f"u = {u}, v = {v}, w = {w}")
print(f"v x w = {cross_product(v, w)}")
print(f"u . (v x w) = {dot(u, cross_product(v, w)):.4f}")
print(f"Volume = {volume:.4f}")

---

## 6. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Compute the cross product $\mathbf{u} \times \mathbf{v}$ where

$$\mathbf{u} = (1, 2, 3), \qquad \mathbf{v} = (4, 5, 6)$$

1. Compute each component by hand using the formula.
2. Verify with `cross_product(u, v)`.
3. Confirm that $\mathbf{u} \times \mathbf{v}$ is **orthogonal** to both
   $\mathbf{u}$ and $\mathbf{v}$ by checking that
   $\mathbf{u} \cdot (\mathbf{u} \times \mathbf{v}) = 0$ and
   $\mathbf{v} \cdot (\mathbf{u} \times \mathbf{v}) = 0$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

A triangle in $\mathbb{R}^3$ has vertices at

$$P = (1, 0, 0), \qquad Q = (0, 2, 0), \qquad R = (0, 0, 3)$$

1. Form edge vectors $\overrightarrow{PQ} = Q - P$ and $\overrightarrow{PR} = R - P$.
2. Compute $\overrightarrow{PQ} \times \overrightarrow{PR}$.
3. The area of the triangle is **half** the parallelogram area:

$$\text{Area}_{\triangle} = \frac{1}{2} \|\overrightarrow{PQ} \times \overrightarrow{PR}\|$$

Compute the triangle's area and verify by computing the area using Heron's formula
(compute all three side lengths, then use
$s = (a+b+c)/2$, $\text{Area} = \sqrt{s(s-a)(s-b)(s-c)}$).

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

A parallelepiped has adjacent edges

$$\mathbf{u} = (1, 2, -1), \qquad \mathbf{v} = (3, 0, 2), \qquad \mathbf{w} = (-1, 1, 4)$$

Compute its volume in **two independent ways**:

1. **Triple scalar product:** $V = |\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w})|$
   using `cross_product` and `dot`.
2. **Determinant:** $V = \left|\det\begin{bmatrix} u_1 & u_2 & u_3 \\ v_1 & v_2 & v_3 \\ w_1 & w_2 & w_3 \end{bmatrix}\right|$
   using `det_cofactor` (and optionally `det_elimination` as a third check).

Verify that both methods give the same answer.

*Bonus:* If the volume is zero, what does that tell you geometrically about
the three vectors?

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Cross product formula** | $\mathbf{u} \times \mathbf{v} = (u_2 v_3 - u_3 v_2,\; u_3 v_1 - u_1 v_3,\; u_1 v_2 - u_2 v_1)$ |
| **Only in $\mathbb{R}^3$** | The cross product takes two 3D vectors and returns a 3D vector |
| **Anticommutative** | $\mathbf{u} \times \mathbf{v} = -(\mathbf{v} \times \mathbf{u})$ — order matters |
| **Perpendicular** | $\mathbf{u} \times \mathbf{v}$ is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$ |
| **Parallelogram area** | $\text{Area} = \|\mathbf{u} \times \mathbf{v}\|$ |
| **Triangle area** | $\text{Area}_{\triangle} = \frac{1}{2}\|\overrightarrow{PQ} \times \overrightarrow{PR}\|$ |
| **Parallelepiped volume** | $V = |\mathbf{u} \cdot (\mathbf{v} \times \mathbf{w})|$ (triple scalar product) |
| **Torque** | $\boldsymbol{\tau} = \mathbf{r} \times \mathbf{F}$ |
| **Determinant connection** | Cross product = formal $3 \times 3$ determinant with $\mathbf{i}, \mathbf{j}, \mathbf{k}$; triple scalar product = $\det$ of the $3 \times 3$ matrix of row vectors |

The cross product ties together inner products, norms, and determinants — three
pillars of this course — into a single operation with deep geometric and physical meaning.